In [ ]:
import os, random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

set_seed(42)

# FOMC Statement Fetch

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# All FOMC statement dates 2006–2026 with SEP flag
# SEP (formerly "Projections") began in April 2011; before that, no SEP meetings
# Format: (date_str, is_sep)
FOMC_MEETINGS = [
    # 2006
    ("20060131", False), ("20060328", False), ("20060510", False),
    ("20060629", False), ("20060920", False), ("20061025", False),
    ("20061212", False),
    # 2007
    ("20070131", False), ("20070321", False), ("20070509", False),
    ("20070618", False), ("20070807", False), ("20070918", False),
    ("20071031", False), ("20071211", False),
    # 2008
    ("20080122", False), ("20080130", False), ("20080318", False),
    ("20080430", False), ("20080625", False), ("20080805", False),
    ("20080916", False), ("20081008", False), ("20081029", False),
    ("20081216", False),
    # 2009
    ("20090128", False), ("20090318", False), ("20090429", False),
    ("20090624", False), ("20090812", False), ("20090923", False),
    ("20091104", False), ("20091216", False),
    # 2010
    ("20100127", False), ("20100316", False), ("20100428", False),
    ("20100623", False), ("20100810", False), ("20100921", False),
    ("20101103", False), ("20101214", False),
    # 2011 — SEP introduced April 2011
    ("20110126", False), ("20110315", False), ("20110427", True),
    ("20110622", True),  ("20110809", False), ("20110921", True),
    ("20111102", False), ("20111213", True),
    # 2012
    ("20120125", True),  ("20120313", False), ("20120425", True),
    ("20120620", True),  ("20120801", False), ("20120913", True),
    ("20121024", False), ("20121212", True),
    # 2013
    ("20130130", False), ("20130320", True),  ("20130501", False),
    ("20130619", True),  ("20130731", False), ("20130918", True),
    ("20131030", False), ("20131218", True),
    # 2014
    ("20140129", False), ("20140319", True),  ("20140430", False),
    ("20140618", True),  ("20140730", False), ("20140917", True),
    ("20141029", False), ("20141217", True),
    # 2015
    ("20150128", False), ("20150318", True),  ("20150429", False),
    ("20150617", True),  ("20150729", False), ("20150917", True),
    ("20151028", False), ("20151216", True),
    # 2016
    ("20160127", False), ("20160316", True),  ("20160427", False),
    ("20160615", True),  ("20160727", False), ("20160921", True),
    ("20161102", False), ("20161214", True),
    # 2017
    ("20170201", False), ("20170315", True),  ("20170503", False),
    ("20170614", True),  ("20170726", False), ("20170920", True),
    ("20171101", False), ("20171213", True),
    # 2018
    ("20180131", False), ("20180321", True),  ("20180502", False),
    ("20180613", True),  ("20180801", False), ("20180926", True),
    ("20181108", False), ("20181219", True),
    # 2019
    ("20190130", False), ("20190320", True),  ("20190501", False),
    ("20190619", True),  ("20190731", False), ("20190918", True),
    ("20191030", False), ("20191211", True),
    # 2020
    ("20200129", False), ("20200303", False), ("20200315", False),
    ("20200429", False), ("20200610", True),  ("20200729", False),
    ("20200916", True),  ("20201105", False), ("20201216", True),
    # 2021
    ("20210127", False), ("20210317", True),  ("20210428", False),
    ("20210616", True),  ("20210728", False), ("20210922", True),
    ("20211103", False), ("20211215", True),
    # 2022
    ("20220126", False), ("20220316", True),  ("20220504", False),
    ("20220615", True),  ("20220727", False), ("20220921", True),
    ("20221102", False), ("20221214", True),
    # 2023
    ("20230201", False), ("20230322", True),  ("20230503", False),
    ("20230614", True),  ("20230726", False), ("20230920", True),
    ("20231101", False), ("20231213", True),
    # 2024
    ("20240131", False), ("20240320", True),  ("20240501", False),
    ("20240612", True),  ("20240731", False), ("20240918", True),
    ("20241107", False), ("20241218", True),
    # 2025
    ("20250129", False), ("20250319", True),  ("20250507", False),
    ("20250618", True),  ("20250730", False), ("20250917", True),
    ("20251029", False), ("20251210", True),
    # 2026
    ("20260128", False), ("20260318", True),
]

BASE_URL = "https://www.federalreserve.gov/newsevents/pressreleases/monetary{}a.htm"

# For pre-2011 statements the URL pattern is the same but content structure
# may differ slightly — add a fallback selector
def fetch_statement(date_str):
    url = BASE_URL.format(date_str)
    r = requests.get(url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    # Try modern layout first, then older layout
    content = (
        soup.find("div", {"class": "col-xs-12 col-sm-8 col-md-8"})
        or soup.find("div", {"id": "content"})
        or soup.find("td", {"class": "content"})   # pre-2015 fallback
        or soup.find("body")
    )
    return content.get_text(separator=" ", strip=True) if content else ""

records = []
for date_str, is_sep in FOMC_MEETINGS:
    print(f"Fetching {date_str}...")
    text = fetch_statement(date_str)
    records.append({
        "date": pd.to_datetime(date_str, format="%Y%m%d"),
        "is_sep": is_sep,
        "raw_text": text,
    })

fomc_df = pd.DataFrame(records).set_index("date").sort_index()
fomc_df.to_parquet("fomc_statements.parquet")

Fetching 20060131...
Fetching 20060328...
Fetching 20060510...
Fetching 20060629...
Fetching 20060920...
Fetching 20061025...
Fetching 20061212...
Fetching 20070131...
Fetching 20070321...
Fetching 20070509...
Fetching 20070618...
Fetching 20070807...
Fetching 20070918...
Fetching 20071031...
Fetching 20071211...
Fetching 20080122...
Fetching 20080130...
Fetching 20080318...
Fetching 20080430...
Fetching 20080625...
Fetching 20080805...
Fetching 20080916...
Fetching 20081008...
Fetching 20081029...
Fetching 20081216...
Fetching 20090128...
Fetching 20090318...
Fetching 20090429...
Fetching 20090624...
Fetching 20090812...
Fetching 20090923...
Fetching 20091104...
Fetching 20091216...
Fetching 20100127...
Fetching 20100316...
Fetching 20100428...
Fetching 20100623...
Fetching 20100810...
Fetching 20100921...
Fetching 20101103...
Fetching 20101214...
Fetching 20110126...
Fetching 20110315...
Fetching 20110427...
Fetching 20110622...
Fetching 20110809...
Fetching 20110921...
Fetching 2011

# Data Cleaning

In [ ]:
import re

# Sections that are boilerplate / non-informative
BOILERPLATE_PATTERNS = [
    r"Voting for the monetary policy action.*",      # vote section
    r"Implementation Note.*",                         # implementation details
    r"For release at.*",                              # release header
    r"\*\s*\*\s*\*",                                  # section dividers
]

def clean_statement(text):
    for pat in BOILERPLATE_PATTERNS:
        text = re.sub(pat, "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

fomc_df["clean_text"] = fomc_df["raw_text"].apply(clean_statement)
fomc_df["text_length"] = fomc_df["clean_text"].apply(lambda t: len(t.split()))

# Signal Creation

In [ ]:
from collections import Counter
import numpy as np

# Curated lexicons — extend these as you see fit
LEXICONS = {
    "hawkish": [
        "raise", "hike", "tighten", "restrictive", "elevated", "persistent",
        "above target", "upside risks", "inflation remains", "further increases",
        "not yet", "remain firm", "additional firming"
    ],
    "dovish": [
        "cut", "lower", "easing", "accommodative", "supportive", "slow",
        "softening", "below target", "downside risks", "pause", "patient",
        "gradual", "modest", "progress toward"
    ],
    "uncertainty": [
        "uncertain", "uncertainty", "risk", "risks", "global", "stress",
        "financial conditions", "volatile", "monitor", "closely watching",
        "remain attentive", "geopolitical"
    ],
    "inflation": [
        "inflation", "price", "prices", "cpi", "pce", "price stability",
        "price pressures", "disinflation", "2 percent", "above 2"
    ],
    "labor": [
        "employment", "unemployment", "labor market", "job", "jobs",
        "payroll", "wage", "wages", "job gains", "labor force",
        "maximum employment", "labor demand"
    ],
}

RATE_DECISION_PATTERNS = {
    "hike":  [r"raise.{0,30}target range", r"increas.{0,30}federal funds",
              r"federal funds rate.{0,30}to \d"],
    "cut":   [r"lower.{0,30}target range", r"decreas.{0,30}federal funds",
              r"reduc.{0,30}target range"],
    "hold":  [r"maintain.{0,30}target range", r"hold.{0,30}federal funds",
              r"leave.{0,30}unchanged"],
}

def score_text(text, keywords):
    tokens = text.lower().split()
    total = len(tokens)
    hits = sum(
        1 for kw in keywords
        for i in range(len(tokens))
        if " ".join(tokens[i:i+len(kw.split())]) == kw.lower()
    )
    return hits / total if total > 0 else 0.0

def detect_policy_decision(text):
    text_lower = text.lower()
    for decision, patterns in RATE_DECISION_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, text_lower):
                return decision
    return "unknown"

for label, keywords in LEXICONS.items():
    fomc_df[f"{label}_score"] = fomc_df["clean_text"].apply(
        lambda t: score_text(t, keywords)
    )

fomc_df["policy_decision"] = fomc_df["clean_text"].apply(detect_policy_decision)

# Encode policy decision as ordinal: cut=-1, hold=0, hike=+1
DECISION_MAP = {"cut": -1, "hold": 0, "hike": 1, "unknown": 0}
fomc_df["policy_encoded"] = fomc_df["policy_decision"].map(DECISION_MAP)

# Embedding

# Cosine Similarity


In [ ]:
# --- Cosine similarity to previous statement ---
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert")
model.eval()

def embed_text(text, strategy="cls"):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    hidden = outputs.last_hidden_state  # (1, seq_len, 768)

    if strategy == "cls":
        emb = hidden[:, 0, :]  # (1, 768)
    else:
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        emb = (hidden * mask).sum(1) / mask.sum(1)  # (1, 768)

    return emb.squeeze(0).cpu().numpy()  # (768,)

print("Generating FinBERT embeddings (takes a few minutes)...")
fomc_df["embedding"] = fomc_df["clean_text"].apply(
    lambda t: embed_text(t, strategy="mean")
)
embedding_matrix = np.stack(fomc_df["embedding"].values)
cos_sims = [1.0]   # first statement has no previous
for i in range(1, len(embedding_matrix)):
    sim = cosine_similarity(
        embedding_matrix[i-1].reshape(1, -1),
        embedding_matrix[i].reshape(1, -1)
    )[0, 0]
    cos_sims.append(sim)

fomc_df["cosine_change_from_prev"] = cos_sims
# High similarity = little change; low = significant language shift
fomc_df["language_shift"] = 1.0 - fomc_df["cosine_change_from_prev"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating FinBERT embeddings (takes a few minutes)...


## Self-Attention

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert")
model.eval()

# Instead of immediately pooling, save ALL token hidden states per statement
def embed_tokens_full(text, max_length=128):
    """
    Returns:
        token_embeddings: (seq_len, 768) — all token hidden states
        attention_mask:   (seq_len,)     — 1 for real tokens, 0 for padding
    max_length=128 is enough: FOMC statements are ~300 words,
    128 tokens covers the substantive content and saves memory.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    with torch.no_grad():
        outputs = model(**inputs)

    token_emb = outputs.last_hidden_state.squeeze(0)       # (128, 768)
    mask      = inputs["attention_mask"].squeeze(0).float() # (128,)
    return token_emb.cpu().numpy(), mask.cpu().numpy()

# Save token embeddings for each statement
import os
os.makedirs("fomc_token_embeddings", exist_ok=True)

for date, row in fomc_df.iterrows():
    date_str = date.strftime("%Y%m%d")
    tok_path  = f"fomc_token_embeddings/{date_str}_tokens.npy"
    mask_path = f"fomc_token_embeddings/{date_str}_mask.npy"

    if os.path.exists(tok_path):   # skip if already computed
        continue

    tokens, mask = embed_tokens_full(row["clean_text"])
    np.save(tok_path,  tokens)
    np.save(mask_path, mask)

print("Done. Token embeddings saved.")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done. Token embeddings saved.


In [ ]:
# import torch
# import torch.nn as nn

# class FOMCTokenAttention(nn.Module):
#     """
#     Learned attention pooling over FinBERT token embeddings.
#     Replaces mean-pooling + PCA.

#     Input:  token_embeddings (batch, 128, 768)
#             attention_mask   (batch, 128)   1=real token, 0=padding
#     Output: statement_vec    (batch, proj_dim)   e.g. 32-dim
#             token_weights    (batch, 128)         for inspection
#     """
#     def __init__(self, token_dim=768, hidden_dim=128, proj_dim=32):
#         super().__init__()

#         # Score each token: is this token relevant for VIX prediction?
#         self.scorer = nn.Sequential(
#             nn.Linear(token_dim, hidden_dim),
#             nn.Tanh(),
#             nn.Linear(hidden_dim, 1, bias=False)
#         )

#         # Project weighted summary to compact embedding
#         self.proj = nn.Sequential(
#             nn.Linear(token_dim, proj_dim),
#             nn.LayerNorm(proj_dim),
#             nn.ReLU(),
#         )

#     def forward(self, token_embeddings, attention_mask):
#         # token_embeddings: (batch, 128, 768)
#         # attention_mask:   (batch, 128)

#         scores = self.scorer(token_embeddings)          # (batch, 128, 1)

#         # Mask padding tokens — set to -inf so softmax zeroes them out
#         mask = attention_mask.unsqueeze(-1)             # (batch, 128, 1)
#         scores = scores.masked_fill(mask == 0, float("-inf"))

#         weights = torch.softmax(scores, dim=1)          # (batch, 128, 1)

#         # Weighted sum across tokens
#         context = (weights * token_embeddings).sum(1)   # (batch, 768)

#         # Project to compact size
#         out = self.proj(context)                        # (batch, proj_dim)

#         return out, weights.squeeze(-1)                 # weights: (batch, 128)

# STMT_PROJ_DIM = 48   # replaces your 10 PCA dims

# fomc_token_attn = FOMCTokenAttention(
#     token_dim=768,
#     hidden_dim=128,
#     proj_dim=STMT_PROJ_DIM
# )
# fomc_token_attn.eval()

# stmt_attn_embeddings = []

# for date, row in fomc_df.iterrows():
#     date_str = date.strftime("%Y%m%d")
#     tokens = np.load(f"fomc_token_embeddings/{date_str}_tokens.npy")  # (128, 768)
#     mask   = np.load(f"fomc_token_embeddings/{date_str}_mask.npy")    # (128,)

#     tok_t  = torch.tensor(tokens).unsqueeze(0)   # (1, 128, 768)
#     mask_t = torch.tensor(mask).unsqueeze(0)     # (1, 128)

#     with torch.no_grad():
#         emb, weights = fomc_token_attn(tok_t, mask_t)

#     stmt_attn_embeddings.append(emb.squeeze(0).numpy())   # (32,)

# stmt_attn_matrix = np.stack(stmt_attn_embeddings)         # (N_statements, 32)

# # Store in fomc_df — replaces stmt_pca_1..10
# for j in range(STMT_PROJ_DIM):
#     fomc_df[f"stmt_attn_{j+1}"] = stmt_attn_matrix[:, j]

## new trained attention

In [175]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import yfinance as yf
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)


# ------------------------------------------------------------
# 2. Download VIX early for FOMC attention supervision
# ------------------------------------------------------------

vix_for_fomc_attn = yf.download(
    "^VIX",
    start="2006-01-01",
    end="2026-04-30",
    progress=False
)["Close"]

# handle yfinance returning dataframe sometimes
if isinstance(vix_for_fomc_attn, pd.DataFrame):
    vix_for_fomc_attn = vix_for_fomc_attn.iloc[:, 0]

vix_for_fomc_attn.name = "^VIX"
vix_for_fomc_attn.index = pd.to_datetime(vix_for_fomc_attn.index).tz_localize(None)
vix_for_fomc_attn = vix_for_fomc_attn.dropna()


# ------------------------------------------------------------
# 3. Build supervised FOMC-level target
#    Target = log(VIX[t+10] / VIX[t])
# ------------------------------------------------------------

def build_fomc_attention_targets(fomc_df, vix_series, horizon=10):
    rows = []

    vix_dates = vix_series.index

    for fomc_date in fomc_df.index:
        fomc_date = pd.Timestamp(fomc_date)

        # use next available VIX trading day if FOMC date is not a trading day
        pos = vix_dates.searchsorted(fomc_date)

        if pos >= len(vix_dates):
            continue

        if pos + horizon >= len(vix_dates):
            continue

        start_date = vix_dates[pos]
        future_date = vix_dates[pos + horizon]

        vix_today = float(vix_series.loc[start_date])
        vix_future = float(vix_series.loc[future_date])

        if vix_today <= 0 or vix_future <= 0:
            continue

        target = np.log(vix_future / vix_today)

        date_str = fomc_date.strftime("%Y%m%d")
        tok_path = f"fomc_token_embeddings/{date_str}_tokens.npy"
        mask_path = f"fomc_token_embeddings/{date_str}_mask.npy"

        # only keep statements whose token embeddings exist
        if not os.path.exists(tok_path) or not os.path.exists(mask_path):
            continue

        rows.append({
            "fomc_date": fomc_date,
            "vix_start_date": start_date,
            "vix_future_date": future_date,
            "fomc_target_10d": target
        })

    out = pd.DataFrame(rows).set_index("fomc_date").sort_index()
    return out


fomc_attn_target_df = build_fomc_attention_targets(
    fomc_df=fomc_df,
    vix_series=vix_for_fomc_attn,
    horizon=10
)

print("FOMC attention training rows:", fomc_attn_target_df.shape)
print(fomc_attn_target_df.head())


# ------------------------------------------------------------
# 4. Chronological train / val split for FOMC attention model
#    We do NOT use the final test-period FOMC statements for training.
# ------------------------------------------------------------

fomc_dates_for_attn = fomc_attn_target_df.index.sort_values()
n_fomc = len(fomc_dates_for_attn)

train_end_fomc = int(n_fomc * 0.80)
val_end_fomc = int(n_fomc * 0.90)

fomc_train_dates = fomc_dates_for_attn[:train_end_fomc]
fomc_val_dates = fomc_dates_for_attn[train_end_fomc:val_end_fomc]
fomc_test_dates = fomc_dates_for_attn[val_end_fomc:]

print("FOMC attn train:", fomc_train_dates[0], "to", fomc_train_dates[-1])
print("FOMC attn val:  ", fomc_val_dates[0], "to", fomc_val_dates[-1])
print("FOMC attn test: ", fomc_test_dates[0], "to", fomc_test_dates[-1])


# ------------------------------------------------------------
# 5. Scale the FOMC attention target using train only
# ------------------------------------------------------------

fomc_y_scaler = StandardScaler()

fomc_y_train_raw = fomc_attn_target_df.loc[
    fomc_train_dates, "fomc_target_10d"
].values.reshape(-1, 1)

fomc_y_scaler.fit(fomc_y_train_raw)

fomc_attn_target_df["fomc_target_10d_scaled"] = fomc_y_scaler.transform(
    fomc_attn_target_df[["fomc_target_10d"]]
).ravel()


# ------------------------------------------------------------
# 6. Dataset
# ------------------------------------------------------------

class FOMCTokenDataset(Dataset):
    def __init__(
        self,
        dates,
        target_df,
        emb_dir="fomc_token_embeddings",
        target_col="fomc_target_10d_scaled"
    ):
        self.dates = list(dates)
        self.target_df = target_df
        self.emb_dir = emb_dir
        self.target_col = target_col

    def __len__(self):
        return len(self.dates)

    def __getitem__(self, idx):
        date = pd.Timestamp(self.dates[idx])
        date_str = date.strftime("%Y%m%d")

        tokens = np.load(f"{self.emb_dir}/{date_str}_tokens.npy").astype(np.float32)
        mask = np.load(f"{self.emb_dir}/{date_str}_mask.npy").astype(np.float32)

        y = np.float32(self.target_df.loc[date, self.target_col])

        return (
            torch.tensor(tokens),      # (128, 768)
            torch.tensor(mask),        # (128,)
            torch.tensor([y])          # (1,)
        )


fomc_train_loader = DataLoader(
    FOMCTokenDataset(fomc_train_dates, fomc_attn_target_df),
    batch_size=8,
    shuffle=True
)

fomc_val_loader = DataLoader(
    FOMCTokenDataset(fomc_val_dates, fomc_attn_target_df),
    batch_size=8,
    shuffle=False
)


# ------------------------------------------------------------
# 7. Trained FOMC token attention model
# ------------------------------------------------------------

class FOMCTokenAttentionRegressor(nn.Module):
    def __init__(
        self,
        token_dim=768,
        hidden_dim=128,
        proj_dim=48,
        dropout=0.20
    ):
        super().__init__()

        self.scorer = nn.Sequential(
            nn.Linear(token_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1, bias=False)
        )

        self.proj = nn.Sequential(
            nn.Linear(token_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = nn.Sequential(
            nn.Linear(proj_dim, 24),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(24, 1)
        )

    def pool(self, token_embeddings, attention_mask):
        # token_embeddings: (batch, 128, 768)
        # attention_mask:   (batch, 128)

        scores = self.scorer(token_embeddings)       # (batch, 128, 1)

        mask = attention_mask.unsqueeze(-1)          # (batch, 128, 1)
        scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)       # (batch, 128, 1)

        context = (weights * token_embeddings).sum(dim=1)   # (batch, 768)
        emb = self.proj(context)                           # (batch, proj_dim)

        return emb, weights.squeeze(-1)

    def forward(self, token_embeddings, attention_mask):
        emb, weights = self.pool(token_embeddings, attention_mask)
        pred = self.head(emb)
        return pred, emb, weights


# ------------------------------------------------------------
# 8. Train FOMC attention model
# ------------------------------------------------------------

def train_fomc_attention_model(
    model,
    train_loader,
    val_loader,
    epochs=300,
    patience=30,
    lr=1e-4,
    weight_decay=1e-4
):
    model = model.to(device)

    criterion = nn.HuberLoss(delta=0.5)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        n_train = 0

        for tokens, mask, y in train_loader:
            tokens = tokens.to(device)
            mask = mask.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            pred, emb, weights = model(tokens, mask)
            loss = criterion(pred, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * tokens.size(0)
            n_train += tokens.size(0)

        train_loss = train_loss / max(n_train, 1)

        model.eval()
        val_loss = 0.0
        n_val = 0

        with torch.no_grad():
            for tokens, mask, y in val_loader:
                tokens = tokens.to(device)
                mask = mask.to(device)
                y = y.to(device)

                pred, emb, weights = model(tokens, mask)
                loss = criterion(pred, y)

                val_loss += loss.item() * tokens.size(0)
                n_val += tokens.size(0)

        val_loss = val_loss / max(n_val, 1)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Val Loss: {val_loss:.5f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


STMT_PROJ_DIM = 48

fomc_token_attn = FOMCTokenAttentionRegressor(
    token_dim=768,
    hidden_dim=128,
    proj_dim=STMT_PROJ_DIM,
    dropout=0.30
)

fomc_token_attn, fomc_attn_history = train_fomc_attention_model(
    model=fomc_token_attn,
    train_loader=fomc_train_loader,
    val_loader=fomc_val_loader,
    epochs=300,
    patience=30,
    lr=1e-3,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# 9. Generate trained FOMC attention embeddings
# ------------------------------------------------------------

fomc_token_attn.eval()

stmt_attn_embeddings = []
stmt_attn_weights = {}

for date, row in fomc_df.iterrows():
    date = pd.Timestamp(date)
    date_str = date.strftime("%Y%m%d")

    tok_path = f"fomc_token_embeddings/{date_str}_tokens.npy"
    mask_path = f"fomc_token_embeddings/{date_str}_mask.npy"

    if not os.path.exists(tok_path) or not os.path.exists(mask_path):
        # fallback zero vector if something is missing
        stmt_attn_embeddings.append(np.zeros(STMT_PROJ_DIM, dtype=np.float32))
        continue

    tokens = np.load(tok_path).astype(np.float32)
    mask = np.load(mask_path).astype(np.float32)

    tok_t = torch.tensor(tokens).unsqueeze(0).to(device)
    mask_t = torch.tensor(mask).unsqueeze(0).to(device)

    with torch.no_grad():
        emb, weights = fomc_token_attn.pool(tok_t, mask_t)

    stmt_attn_embeddings.append(emb.squeeze(0).cpu().numpy())
    stmt_attn_weights[date] = weights.squeeze(0).cpu().numpy()

stmt_attn_matrix = np.stack(stmt_attn_embeddings)

# remove old stmt_attn columns if rerunning this cell
old_stmt_cols = [c for c in fomc_df.columns if c.startswith("stmt_attn_")]
fomc_df = fomc_df.drop(columns=old_stmt_cols, errors="ignore")

for j in range(STMT_PROJ_DIM):
    fomc_df[f"stmt_attn_{j+1}"] = stmt_attn_matrix[:, j]

print("Added trained stmt_attn columns:")
print([f"stmt_attn_{j+1}" for j in range(STMT_PROJ_DIM)])
print("fomc_df shape:", fomc_df.shape)

Using device: cuda
FOMC attention training rows: (164, 3)
           vix_start_date vix_future_date  fomc_target_10d
fomc_date                                                 
2006-01-31     2006-01-31      2006-02-14        -0.055570
2006-03-28     2006-03-28      2006-04-11         0.115670
2006-05-10     2006-05-10      2006-05-24         0.387766
2006-06-29     2006-06-29      2006-07-14         0.325891
2006-09-20     2006-09-20      2006-10-04         0.040436
FOMC attn train: 2006-01-31 00:00:00 to 2022-01-26 00:00:00
FOMC attn val:   2022-03-16 00:00:00 to 2024-01-31 00:00:00
FOMC attn test:  2024-03-20 00:00:00 to 2026-03-18 00:00:00


/tmp/ipykernel_12492/4008199934.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix_for_fomc_attn = yf.download(
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Epoch 001 | Train Loss: 0.26725 | Val Loss: 0.20500
Epoch 002 | Train Loss: 0.26377 | Val Loss: 0.20017
Epoch 003 | Train Loss: 0.26727 | Val Loss: 0.19745
Epoch 004 | Train Loss: 0.26033 | Val Loss: 0.19949
Epoch 005 | Train Loss: 0.26014 | Val Loss: 0.20054
Epoch 006 | Train Loss: 0.26336 | Val Loss: 0.19400
Epoch 007 | Train Loss: 0.25728 | Val Loss: 0.19550
Epoch 008 | Train Loss: 0.26053 | Val Loss: 0.19943
Epoch 009 | Train Loss: 0.25351 | Val Loss: 0.19040
Epoch 010 | Train Loss: 0.25629 | Val Loss: 0.19289
Epoch 011 | Train Loss: 0.26346 | Val Loss: 0.21650
Epoch 012 | Train Loss: 0.25353 | Val Loss: 0.19986
Epoch 013 | Train Loss: 0.24307 | Val Loss: 0.20520
Epoch 014 | Train Loss: 0.22311 | Val Loss: 0.23271
Epoch 015 | Train Loss: 0.24388 | Val Loss: 0.22905
Epoch 016 | Train Loss: 0.24520 | Val Loss: 0.22379
Epoch 017 | Train Loss: 0.21596 | Val Loss: 0.22175
Epoch 018 | Train Loss: 0.22667 | Val Loss: 0.23312
Epoch 019 | Train Loss: 0.22010 | Val Loss: 0.20579
Epoch 020 | 

## Daily FOMC Calendar Features


In [176]:
daily_dates = pd.bdate_range("2006-01-01", "2026-04-30")  # business days
fomc_dates_sorted = fomc_df.index.sort_values()

def build_daily_fomc_features(daily_dates, fomc_df):
    rows = []
    # score_cols =  [
    #     "hawkish_score", "dovish_score", "uncertainty_score",
    #     "inflation_score", "labor_score", "policy_encoded",
    #     "text_length", "cosine_change_from_prev", "language_shift",
    #     "is_sep",] + [f"stmt_attn_{j+1}" for j in range(STMT_PROJ_DIM)]
        # [f"change_pca_{i+1}" for i in range(N_COMPONENTS)]
      #    + [f"stmt_pca_{i+1}" for i in range(N_COMPONENTS)] \

    score_cols =  [
    "hawkish_score", "dovish_score", "uncertainty_score",
    "inflation_score", "labor_score", "policy_encoded",
    "text_length", "cosine_change_from_prev", "language_shift",
    "is_sep",] + [f"stmt_attn_{j+1}" for j in range(STMT_PROJ_DIM)]

    last_features = {col: 0.0 for col in score_cols}
    last_fomc_date = None
    fomc_dates_list = list(fomc_df.index)

    for day in daily_dates:
        # Update carry-forward if today is an FOMC day
        is_fomc_day = int(day in fomc_df.index)
        if is_fomc_day:
            last_features = {col: fomc_df.loc[day, col] for col in score_cols}
            last_fomc_date = day

        # Days since last FOMC
        if last_fomc_date is not None:
            days_since = (day - last_fomc_date).days
        else:
            days_since = np.nan

        # Days to next FOMC
        future = [d for d in fomc_dates_list if d > day]
        days_to_next = (future[0] - day).days if future else np.nan

        row = {
            "date": day,
            "is_fomc_day": is_fomc_day,
            "days_since_last_fomc": days_since,
            "days_to_next_fomc": days_to_next,
            **last_features,
        }
        rows.append(row)

    return pd.DataFrame(rows).set_index("date")

daily_fomc = build_daily_fomc_features(daily_dates, fomc_df)
# Fix boolean columns before saving
daily_fomc["is_sep"] = daily_fomc["is_sep"].astype(bool)
daily_fomc["is_fomc_day"] = daily_fomc["is_fomc_day"].astype(int)

daily_fomc.to_parquet("daily_fomc_features.parquet")

# Google Trends

In [ ]:
 !pip install pytrends gdeltdoc

In [ ]:
from pytrends.request import TrendReq
import pandas as pd
import numpy as np
import time

def fetch_google_trends(
    keywords,
    start="2006-01-01",
    end="2026-04-30",
    geo="US",
    sleep_sec=2
):
    """
    Returns weekly Google Trends data.
    Google Trends long-horizon data is usually weekly, not daily.
    We will forward-fill to business-day frequency later.
    """
    pytrends = TrendReq(hl="en-US", tz=360)

    pytrends.build_payload(
        kw_list=keywords,
        timeframe=f"{start} {end}",
        geo=geo
    )

    df = pytrends.interest_over_time()

    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])

    df.index = pd.to_datetime(df.index).tz_localize(None)
    df = df.astype(float)

    time.sleep(sleep_sec)
    return df


trend_keywords = [
    "recession",
    "market crash",
    "inflation",
    "unemployment",
    "VIX"
]

google_trends = fetch_google_trends(
    trend_keywords,
    start="2006-01-01",
    end="2026-04-30",
    geo="US"
)

# Rename columns clearly
google_trends = google_trends.rename(columns={
    "recession": "gt_recession",
    "market crash": "gt_market_crash",
    "inflation": "gt_inflation",
    "unemployment": "gt_unemployment",
    "VIX": "gt_vix"
})

# Convert weekly trends to business-day features
google_trends_daily = (
    google_trends
    .reindex(daily_dates)
    .ffill()
)

# Feature engineering: changes and z-scores
for col in google_trends_daily.columns:
    google_trends_daily[f"{col}_chg"] = google_trends_daily[col].diff()

    google_trends_daily[f"{col}_z"] = (
        (google_trends_daily[col] - google_trends_daily[col].rolling(60).mean())
        / google_trends_daily[col].rolling(60).std()
    )

google_trends_daily = google_trends_daily.replace([np.inf, -np.inf], np.nan)
google_trends_daily = google_trends_daily.ffill()

google_trends_daily.to_parquet("google_trends_features.parquet")

print(google_trends_daily.head())
print(google_trends_daily.columns.tolist())

            gt_recession  gt_market_crash  gt_inflation  gt_unemployment  \
2006-01-02           NaN              NaN           NaN              NaN   
2006-01-03           NaN              NaN           NaN              NaN   
2006-01-04           NaN              NaN           NaN              NaN   
2006-01-05           NaN              NaN           NaN              NaN   
2006-01-06           NaN              NaN           NaN              NaN   

            gt_vix  gt_recession_chg  gt_recession_z  gt_market_crash_chg  \
2006-01-02     NaN               NaN             NaN                  NaN   
2006-01-03     NaN               NaN             NaN                  NaN   
2006-01-04     NaN               NaN             NaN                  NaN   
2006-01-05     NaN               NaN             NaN                  NaN   
2006-01-06     NaN               NaN             NaN                  NaN   

            gt_market_crash_z  gt_inflation_chg  gt_inflation_z  \
2006-01-02   

# Merge with VIX + Unemployment

In [177]:
import yfinance as yf

import pandas as pd

# VIX
vix = yf.download("^VIX", start="2006-01-01", end="2026-04-30")["Close"]
vix.name = "^VIX"
vix.index = pd.to_datetime(vix.index).tz_localize(None)

# # Unemployment (monthly → forward-fill to daily)
# unrate = web.DataReader("UNRATE", "fred", "2006-01-01", "2026-04-30")
# unrate = unrate.reindex(daily_dates).ffill()
# unrate.columns = ["unemployment"]

# Merge everything
# define master index first
master = daily_fomc.copy()

# reindex VIX
vix = vix.reindex(master.index)

# reindex Google Trends
google_trends_daily = google_trends_daily.reindex(master.index)

# now join
master = master.join(vix, how="left")
master = master.join(google_trends_daily, how="left")

# fill
master = master.ffill().bfill()


# # Target variable options:
# master["vix_10d_change"]  = (( master["^VIX"].shift(-10)/master["^VIX"]))-1
master["target_vix_log_change_10d"] = np.log(master["^VIX"].shift(-10) / master["^VIX"])
# # master["vix_5d_vol"]  = master['^VIX'].rolling(-5).std()

# Time Series Feature
master["vix_lag1"]     = master["^VIX"].shift(1)
master["vix_lag5"]     = master["^VIX"].shift(5)
master["vix_ma10"]     = master["^VIX"].rolling(10).mean()
master["vix_ma30"]     = master["^VIX"].rolling(30).mean()
master["vix_zscore"]   = (
    (master["^VIX"] - master["^VIX"].rolling(60).mean())
    / master["^VIX"].rolling(60).std()
)
# VIX term structure proxy — current vs recent average
master["vix_spread"]   = master["^VIX"] - master["^VIX"].rolling(20).mean()

# Realized vol of VIX itself (vol-of-vol)
master["vix_realized_vol_20d"] = (
    np.log(master["^VIX"] / master["^VIX"].shift(1))
    .rolling(20).std()
)



/tmp/ipykernel_12492/997604267.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX", start="2006-01-01", end="2026-04-30")["Close"]
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  1 of 1 completed


In [178]:
master.shape

(5304, 85)

In [179]:

target_cols = ["target_vix_log_change_10d"]

master = master.dropna(subset=target_cols)
master.to_parquet("master_features.parquet")
print(f"Final dataset: {master.shape}")
print(master.columns.tolist())

Final dataset: (5294, 85)
['is_fomc_day', 'days_since_last_fomc', 'days_to_next_fomc', 'hawkish_score', 'dovish_score', 'uncertainty_score', 'inflation_score', 'labor_score', 'policy_encoded', 'text_length', 'cosine_change_from_prev', 'language_shift', 'is_sep', 'stmt_attn_1', 'stmt_attn_2', 'stmt_attn_3', 'stmt_attn_4', 'stmt_attn_5', 'stmt_attn_6', 'stmt_attn_7', 'stmt_attn_8', 'stmt_attn_9', 'stmt_attn_10', 'stmt_attn_11', 'stmt_attn_12', 'stmt_attn_13', 'stmt_attn_14', 'stmt_attn_15', 'stmt_attn_16', 'stmt_attn_17', 'stmt_attn_18', 'stmt_attn_19', 'stmt_attn_20', 'stmt_attn_21', 'stmt_attn_22', 'stmt_attn_23', 'stmt_attn_24', 'stmt_attn_25', 'stmt_attn_26', 'stmt_attn_27', 'stmt_attn_28', 'stmt_attn_29', 'stmt_attn_30', 'stmt_attn_31', 'stmt_attn_32', 'stmt_attn_33', 'stmt_attn_34', 'stmt_attn_35', 'stmt_attn_36', 'stmt_attn_37', 'stmt_attn_38', 'stmt_attn_39', 'stmt_attn_40', 'stmt_attn_41', 'stmt_attn_42', 'stmt_attn_43', 'stmt_attn_44', 'stmt_attn_45', 'stmt_attn_46', 'stmt_attn


 # Train Model

In [180]:
!pip install arch

## Baseline - GARCH

In [181]:
import numpy as np
from arch import arch_model
vix = master["^VIX"].copy()

# log returns
ret = np.log(vix / vix.shift(1)).dropna()

# GARCH typically expects percentage scale
ret = ret * 100
train_size = int(len(ret) * 0.8)

train = ret.iloc[:train_size]
test = ret.iloc[train_size:]

model = arch_model(
    train,
    mean="Constant",     # mean model
    vol="GARCH",
    p=1,
    q=1,
    dist="normal"
)

res = model.fit(disp="off")

print(res.summary())


                     Constant Mean - GARCH Model Results                      
Dep. Variable:                   ^VIX   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -14345.0
Distribution:                  Normal   AIC:                           28698.0
Method:            Maximum Likelihood   BIC:                           28723.4
                                        No. Observations:                 4234
Date:                Mon, May 04 2026   Df Residuals:                     4233
Time:                        17:41:55   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu            -0.0145  9.945e-02     -0.146      0.884 [ -0.209,  0.18

In [182]:
horizon = len(test)

forecast = res.forecast(horizon=horizon)

# Extract variance forecast
variance = forecast.variance.values[-1]

# Convert to volatility
vol_forecast = np.sqrt(variance)

vol_forecast = pd.Series(vol_forecast, index=test.index)

# realized volatility (10-day)
realized_vol = (
    ret.rolling(10).std().shift(-10)
).loc[test.index]

from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(realized_vol.dropna(), vol_forecast.loc[realized_vol.dropna().index]))

print("GARCH RMSE:", rmse)

GARCH RMSE: 3.8091924013212526


## LSTM

In [183]:
TARGET_COL = "target_vix_log_change_10d"
# Drop all dates before the MIN_STATEMENTS-th FOMC meeting
# burnin_cutoff = fomc_df.index[MIN_STATEMENTS]  # 13th FOMC date (0-indexed)
# master_model = master.loc[burnin_cutoff:]
# master_model = master_model.replace([np.inf, -np.inf], np.nan).dropna().copy()
master_model = master.copy()
master_model = master_model.replace([np.inf, -np.inf], np.nan).dropna().copy()
# keeps numeric features only and excludes target columns.
exclude_cols = [
    "target_vix_log_change_10d"
]

feature_cols = [
    col for col in master_model.columns
    if col not in exclude_cols
]

X_df = master_model[feature_cols].select_dtypes(include=[np.number, "bool"]).copy()
X_df = X_df.astype(float)

y = master_model[TARGET_COL].copy()

print("Number of rows:", len(X_df))
print("Number of features:", X_df.shape[1])
print("Feature columns:")
print(X_df.columns.tolist())

Number of rows: 5235
Number of features: 84
Feature columns:
['is_fomc_day', 'days_since_last_fomc', 'days_to_next_fomc', 'hawkish_score', 'dovish_score', 'uncertainty_score', 'inflation_score', 'labor_score', 'policy_encoded', 'text_length', 'cosine_change_from_prev', 'language_shift', 'is_sep', 'stmt_attn_1', 'stmt_attn_2', 'stmt_attn_3', 'stmt_attn_4', 'stmt_attn_5', 'stmt_attn_6', 'stmt_attn_7', 'stmt_attn_8', 'stmt_attn_9', 'stmt_attn_10', 'stmt_attn_11', 'stmt_attn_12', 'stmt_attn_13', 'stmt_attn_14', 'stmt_attn_15', 'stmt_attn_16', 'stmt_attn_17', 'stmt_attn_18', 'stmt_attn_19', 'stmt_attn_20', 'stmt_attn_21', 'stmt_attn_22', 'stmt_attn_23', 'stmt_attn_24', 'stmt_attn_25', 'stmt_attn_26', 'stmt_attn_27', 'stmt_attn_28', 'stmt_attn_29', 'stmt_attn_30', 'stmt_attn_31', 'stmt_attn_32', 'stmt_attn_33', 'stmt_attn_34', 'stmt_attn_35', 'stmt_attn_36', 'stmt_attn_37', 'stmt_attn_38', 'stmt_attn_39', 'stmt_attn_40', 'stmt_attn_41', 'stmt_attn_42', 'stmt_attn_43', 'stmt_attn_44', 'stmt_a

In [184]:

# Time-series split
dates = X_df.index
n = len(X_df)
train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]

print("Train:", train_dates[0], "to", train_dates[-1])
print("Val:  ", val_dates[0], "to", val_dates[-1])
print("Test: ", test_dates[0], "to", test_dates[-1])

Train: 2006-03-24 00:00:00 to 2022-04-12 00:00:00
Val:   2022-04-13 00:00:00 to 2024-04-12 00:00:00
Test:  2024-04-15 00:00:00 to 2026-04-16 00:00:00


In [185]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_raw = X_df.loc[train_dates]
y_train_raw = y.loc[train_dates]

scaler_X.fit(X_train_raw)
scaler_y.fit(y_train_raw.values.reshape(-1, 1))

X_scaled = pd.DataFrame(
    scaler_X.transform(X_df),
    index=X_df.index,
    columns=X_df.columns
)

y_scaled = pd.Series(
    scaler_y.transform(y.values.reshape(-1, 1)).ravel(),
    index=y.index,
    name=TARGET_COL
)

# last 60 trading days of features → target at current day
def make_lstm_sequences(X_df, y_series, seq_len=60):
    X_values = X_df.values
    y_values = y_series.values
    dates = X_df.index

    X_seq = []
    y_seq = []
    target_dates = []

    for i in range(seq_len, len(X_df)):
        X_seq.append(X_values[i - seq_len:i])
        y_seq.append(y_values[i])
        target_dates.append(dates[i])

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(y_seq, dtype=np.float32),
        np.array(target_dates)
    )


SEQ_LEN = 90 #360

X_seq, y_seq, seq_dates = make_lstm_sequences(
    X_scaled,
    y_scaled,
    seq_len=SEQ_LEN
)

print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)
print("Date range:", seq_dates[0], "to", seq_dates[-1])

# split sequences by their target date:
train_mask = seq_dates <= train_dates[-1]
val_mask = (seq_dates > train_dates[-1]) & (seq_dates <= val_dates[-1])
test_mask = seq_dates > val_dates[-1]

X_train, y_train = X_seq[train_mask], y_seq[train_mask]
X_val, y_val = X_seq[val_mask], y_seq[val_mask]
X_test, y_test = X_seq[test_mask], y_seq[test_mask]

test_seq_dates = seq_dates[test_mask]

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

X_seq shape: (5145, 90, 84)
y_seq shape: (5145,)
Date range: 2006-07-28 00:00:00 to 2026-04-16 00:00:00
Train: (4098, 90, 84) (4098,)
Val:   (523, 90, 84) (523,)
Test:  (524, 90, 84) (524,)


In [186]:
# PyTorch Dataset and DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

class VIXSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64

train_loader = DataLoader(
    VIXSequenceDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    VIXSequenceDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    VIXSequenceDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)

## LSTM W/O attention

In [187]:
import torch.nn as nn
# The LSTM reads the 60-day sequence. The final hidden state is sent into an MLP.
class VIXLSTMMLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        num_layers=1,
        mlp_hidden_dim=16,
        dropout=0.10,
    ):
        super().__init__()



        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, 1)
        )

    def forward(self, x):
        # x: [batch, seq_len, num_features]
        lstm_out, (h_n, c_n) = self.lstm(x)

        # last hidden state from final LSTM layer
        h_last = h_n[-1]  # [batch, hidden_dim]

        pred = self.mlp(h_last)  # [batch, 1]
        return pred

## LSTM With Attention

In [188]:
import torch
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_dim)
        scores  = self.score(lstm_out)              # (batch, seq_len, 1)
        weights = torch.softmax(scores, dim=1)      # (batch, seq_len, 1)
        context = (weights * lstm_out).sum(dim=1)   # (batch, hidden_dim)
        return context, weights                     # return weights for inspection


class VIXLSTMMLP_ATT(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        num_layers=1,
        mlp_hidden_dim=32,
        dropout=0.3,
    ):
        super().__init__()

        # Normalize across feature dimension at each timestep
        # Stabilizes gradient flow when features have different scales
        self.input_norm = nn.LayerNorm(input_dim)

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,         # inter-layer dropout (only applies when num_layers > 1)
        )

        # Attention over all 60 timesteps
        self.attention = Attention(hidden_dim)

        # Post-LSTM dropout
        self.dropout = nn.Dropout(dropout)

        # Input is attention context + final hidden state concatenated
        # so MLP input dim is hidden_dim * 2
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, mlp_hidden_dim),
            nn.LayerNorm(mlp_hidden_dim),            # normalize before activation
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, mlp_hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim // 2, 1)
        )

    def forward(self, x, return_attention=False):
        # x: (batch, seq_len=60, input_dim)

        # Step 1: normalize input features at each timestep
        x = self.input_norm(x)

        # Step 2: LSTM over full 60-day sequence
        lstm_out, (h_n, c_n) = self.lstm(x)
        # lstm_out: (batch, 60, hidden_dim) — all timestep hidden states
        # h_n:      (num_layers, batch, hidden_dim)

        # Step 3: attention context — weighted sum of all 60 hidden states
        # lets the model upweight FOMC event days automatically
        context, attn_weights = self.attention(lstm_out)
        # context: (batch, hidden_dim)

        # Step 4: final hidden state from last LSTM layer
        h_last = h_n[-1]   # (batch, hidden_dim)

        # Step 5: concatenate both — attention captures "what mattered"
        # while h_last captures "most recent state"
        combined = torch.cat([context, h_last], dim=-1)  # (batch, hidden_dim * 2)
        combined = self.dropout(combined)

        # Step 6: MLP → scalar prediction
        pred = self.mlp(combined)   # (batch, 1)

        if return_attention:
            return pred, attn_weights
        return pred

## TBPTT

In [189]:
import torch
import torch.nn as nn

class VIXLSTM_TBPTT(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        num_layers=1,
        mlp_hidden_dim=16,
        dropout=0.2,
        tbptt_step=None,
    ):
        super().__init__()
        self.tbptt_step = tbptt_step
        self.input_norm = nn.LayerNorm(input_dim)

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, 1)
        )

    def forward(self, x):
        """
        x: (batch, seq_len, input_dim)

        tbptt_step=None → full BPTT
        tbptt_step=10   → truncate gradient every 10 timesteps
        """

        x = self.input_norm(x)

        # Full BPTT baseline
        if self.tbptt_step is None:
            _, (h, c) = self.lstm(x)
            h_last = h[-1]
            return self.mlp(h_last)

        # TBPTT
        h, c = None, None
        seq_len = x.size(1)

        for start in range(0, seq_len, self.tbptt_step):
            end = min(start + self.tbptt_step, seq_len)
            x_chunk = x[:, start:end, :]

            if h is None:
                _, (h, c) = self.lstm(x_chunk)
            else:
                _, (h, c) = self.lstm(x_chunk, (h, c))

            # truncate gradient history
            h = h.detach()
            c = c.detach()

        h_last = h[-1]
        return self.mlp(h_last)

# Old Experiments

In [190]:
import torch.optim as optim
# Train with early stopping; Use Huber loss because VIX has extreme jumps.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

model = VIXLSTMMLP_ATT( #VIXLSTMMLP, VIXLSTMMLP_ATT, VIXLSTM_TBPTT
    input_dim=X_train.shape[2],
    hidden_dim=32, #32,
    num_layers=1,
    mlp_hidden_dim=16,
    dropout=0.3,
    # tbptt_step=3,
).to(device)


criterion = nn.HuberLoss(delta=0.3) #delta = 1, 0.5
optimizer = optim.AdamW(model.parameters(), lr=0.00008, weight_decay=1e-4)


def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n_obs = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            pred = model(X_batch)
            loss = criterion(pred, y_batch)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        n_obs += batch_size

    return total_loss / n_obs


EPOCHS = 400
PATIENCE = 15

best_val_loss = float("inf")
best_state = None
patience_counter = 0

history = []



for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, train=True)
    val_loss = run_epoch(model, val_loader, train=False)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_state)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 001 | Train Loss: 0.18335 | Val Loss: 0.12548
Epoch 002 | Train Loss: 0.18161 | Val Loss: 0.12511
Epoch 003 | Train Loss: 0.17939 | Val Loss: 0.12484
Epoch 004 | Train Loss: 0.17847 | Val Loss: 0.12475
Epoch 005 | Train Loss: 0.17731 | Val Loss: 0.12475
Epoch 006 | Train Loss: 0.17751 | Val Loss: 0.12481
Epoch 007 | Train Loss: 0.17577 | Val Loss: 0.12464
Epoch 008 | Train Loss: 0.17546 | Val Loss: 0.12460
Epoch 009 | Train Loss: 0.17492 | Val Loss: 0.12432
Epoch 010 | Train Loss: 0.17528 | Val Loss: 0.12440
Epoch 011 | Train Loss: 0.17380 | Val Loss: 0.12487
Epoch 012 | Train Loss: 0.17302 | Val Loss: 0.12549
Epoch 013 | Train Loss: 0.17288 | Val Loss: 0.12611
Epoch 014 | Train Loss: 0.17255 | Val Loss: 0.12725
Epoch 015 | Train Loss: 0.17171 | Val Loss: 0.12715
Epoch 016 | Train Loss: 0.17057 | Val Loss: 0.12820
Epoch 017 | Train Loss: 0.17093 | Val Loss: 0.12961
Epoch 018 | Train Loss: 0.17043 | Val Loss: 0.13121
Epoch 019 | Train Loss: 0.16969 | Val Loss: 0.13213
Epoch 020 | 

<All keys matched successfully>

In [191]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Evaluate on test set
def predict_loader(model, loader):
    model.eval()

    preds = []
    actuals = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)

            pred = model(X_batch).cpu().numpy().ravel()
            actual = y_batch.numpy().ravel()

            preds.append(pred)
            actuals.append(actual)

    preds = np.concatenate(preds)
    actuals = np.concatenate(actuals)

    return preds, actuals


pred_scaled, y_true_scaled = predict_loader(model, test_loader)

# Convert back to original target scale
pred = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
y_true = scaler_y.inverse_transform(y_true_scaled.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_true, pred)
rmse = np.sqrt(mean_squared_error(y_true, pred))
r2 = r2_score(y_true, pred)
direction_acc = np.mean((pred > 0) == (y_true > 0))

print("Test MAE:", round(mae, 5))
print("Test RMSE:", round(rmse, 5))
print("Test R2:", round(r2, 5))
print("Directional Accuracy:", round(direction_acc, 4))

Test MAE: 0.15629
Test RMSE: 0.21489
Test R2: 0.04855
Directional Accuracy: 0.5592


In [192]:
# Baseline 1: predict zero change
baseline_zero = np.zeros_like(y_true)

zero_mae = mean_absolute_error(y_true, baseline_zero)
zero_rmse = np.sqrt(mean_squared_error(y_true, baseline_zero))
zero_direction_acc = np.mean((baseline_zero > 0) == (y_true > 0))

print("Zero-change baseline MAE:", round(zero_mae, 5))
print("Zero-change baseline RMSE:", round(zero_rmse, 5))
print("Zero-change baseline directional accuracy:", round(zero_direction_acc, 4))

Zero-change baseline MAE: 0.15879
Zero-change baseline RMSE: 0.22031
Zero-change baseline directional accuracy: 0.4828


In [193]:
# Baseline 2: predict train mean
train_target_original = scaler_y.inverse_transform(
    y_train.reshape(-1, 1)
).ravel()

train_mean = train_target_original.mean()
baseline_mean = np.full_like(y_true, train_mean)
baseline_mean_direction_acc = np.mean((baseline_mean > 0) == (y_true > 0))

mean_mae = mean_absolute_error(y_true, baseline_mean)
mean_rmse = np.sqrt(mean_squared_error(y_true, baseline_mean))

print("Train-mean baseline MAE:", round(mean_mae, 5))
print("Train-mean baseline RMSE:", round(mean_rmse, 5))
print("Train-mean baseline directional accuracy:", round(baseline_mean_direction_acc, 4))

Train-mean baseline MAE: 0.15875
Train-mean baseline RMSE: 0.22031
Train-mean baseline directional accuracy: 0.5172
